
## Establish the baseline
1. Create notebooks/00-baseline.ipynb . Download one month of NYC TLC green taxi trip data
(Parquet).

2. Build the minimum viable model: engineer PU_DO (pickup–dropoff pair) and trip_distance , use DictVectorizer + LinearRegression or RandomForestRegressor , target is duration in
minutes.

3. Print validation RMSE and MAE. Write both at the top of reports/module-1.md .

4. Save the fitted model to models/baseline.pkl .

This notebook is deliberately messy. It is the "before" picture — commit it as-is and point at it in your report to show what you replaced.



### Download one month of NYC TLC green taxi trip data(Parquet).

In [1]:
import pickle
import numpy as np
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split


In [2]:
df = pd.read_parquet('/content/datasets/green_tripdata_2026-02.parquet')

In [3]:
df.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,2,2026-02-01 00:58:25,2026-02-01 01:03:26,N,1.0,82,129,1.0,0.50,6.5,...,0.5,1.80,0.0,NaN,1.0,10.80,1.0,1.0,0.0,0.00
1,2,2026-02-01 00:05:50,2026-02-01 00:13:19,N,1.0,33,65,1.0,1.08,9.3,...,0.5,2.95,0.0,NaN,1.0,14.75,1.0,1.0,0.0,0.00
2,1,2026-02-01 00:45:54,2026-02-01 01:28:27,N,5.0,226,143,1.0,5.20,75.0,...,0.0,22.70,0.0,NaN,0.0,98.45,1.0,2.0,0.0,0.75
3,2,2026-02-01 00:06:38,2026-02-01 00:15:56,N,1.0,74,235,1.0,4.70,20.5,...,0.5,0.00,0.0,NaN,1.0,23.00,2.0,1.0,0.0,0.00
4,2,2026-02-01 00:37:31,2026-02-01 00:51:10,N,1.0,120,119,1.0,2.47,16.3,...,0.5,0.00,0.0,NaN,1.0,18.80,2.0,1.0,0.0,0.00


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37373 entries, 0 to 37372
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   VendorID               37373 non-null  int32         
 1   lpep_pickup_datetime   37373 non-null  datetime64[us]
 2   lpep_dropoff_datetime  37373 non-null  datetime64[us]
 3   store_and_fwd_flag     31986 non-null  object        
 4   RatecodeID             31986 non-null  float64       
 5   PULocationID           37373 non-null  int32         
 6   DOLocationID           37373 non-null  int32         
 7   passenger_count        31986 non-null  float64       
 8   trip_distance          37373 non-null  float64       
 9   fare_amount            37373 non-null  float64       
 10  extra                  37373 non-null  float64       
 11  mta_tax                37373 non-null  float64       
 12  tip_amount             37373 non-null  float64       
 13  t

In [5]:
df["duration"] = df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]
df["duration"] = df["duration"].dt.total_seconds() / 60

In [6]:
df["duration"].describe(percentiles=[0.5,0.75,0.95, 0.99])

,duration
count,37373.000000
mean,20.707001
std,72.301272
min,-10.183333
50%,13.316667
75%,20.616667
95%,43.866667
99%,82.640000
max,1438.316667


In [7]:
len(df[df['duration']>60])

872

In [8]:
df = df[(df["duration"] >= 1) & (df["duration"] <= 60)]

In [9]:
df["trip_distance"].describe(percentiles=[0.5,0.75,0.95, 0.99])

,trip_distance
count,35319.000000
mean,22.446717
std,1193.604970
min,0.000000
50%,1.990000
75%,3.370000
95%,8.761000
99%,14.860000
max,117566.000000


In [10]:
len(df[df['trip_distance']>20])

86

In [11]:
df = df[(df["trip_distance"] > 0) & (df["trip_distance"] <= 20)]

In [12]:
df["PU_DO"] = df["PULocationID"].astype(str) + "_" + df["DOLocationID"].astype(str)

In [13]:
df=df[['PU_DO','trip_distance','duration']]

In [14]:
df.isnull().sum()

,0
PU_DO,0
trip_distance,0
duration,0


In [15]:
X=df.drop('duration',axis=1)
y=df['duration']

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [17]:
dv = DictVectorizer()
train_dicts = X_train.to_dict(orient="records")
X_train_vectorized = dv.fit_transform(train_dicts)
test_dicts = X_test.to_dict(orient="records")
X_test_vectorized = dv.transform(test_dicts)

In [18]:
lr = LinearRegression()
lr.fit(X_train_vectorized, y_train)

LinearRegression()

In [19]:
y_pred = lr.predict(X_test_vectorized)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"Validation RMSE: {rmse:.2f}")
print(f"Validation MAE: {mae:.2f}")

Validation RMSE: 6.55
Validation MAE: 4.29


In [20]:
import os
os.makedirs("reports", exist_ok=True)
with open("reports/module-1.md", "w") as f:
    f.write("# 📊 Module 1 — Baseline Report\n\n")
    f.write("## ✅ Validation Metrics\n\n")
    f.write("| Metric | Value |\n")
    f.write("|--------|-------|\n")
    f.write(f"| RMSE   | {rmse:.2f} |\n")
    f.write(f"| MAE    | {mae:.2f} |\n\n")
    f.write("## ⚙️ Setup\n\n")
    f.write("| | |\n")
    f.write("|---|---|\n")
    f.write("| **Data** | NYC TLC Green Taxi trip data (Parquet) — one month train / one month validation |\n")
    f.write("| **Features** | `PU_DO` (pickup–dropoff pair), `trip_distance` |\n")
    f.write("| **Target** | trip `duration` (minutes) |\n")
    f.write("| **Vectorizer** | DictVectorizer |\n")
    f.write("| **Model** | LinearRegression |\n\n")
    f.write("## 📝 Notes\n\n")
    f.write("> This is the baseline notebook (`notebooks/00-baseline.ipynb`), kept deliberately messy — ")
    f.write("the *\"before\"* picture. It will be replaced with a structured, tested, packaged version ")
    f.write("in the following steps of Module 1.\n")

In [21]:
with open('/content/models/lin_reg.pkl', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)